In [2]:
# 로컬 방식으로 STT
import os
import sys
import warnings


import transformers # LLM
from transformers import AutoProcessor


warnings.filterwarnings('ignore') # 경고 무시


c:\big21\agentic_ai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ffmpeg 파이썬에서 사용할 수 있도록 환경 설정 필요
# DLL: 윈도우 실행 파일 => 등록해야 파이썬에서 호출, 실행 가능
# 실행 파일 직접 등록할 것.
ffmpeg_path=r"C:\big21\agentic_ai\chap05\ffmpeg-7.1.1-full_build-shared\bin"

# 1. 해당 파일 존재 여부 확인
if os.path.exists(ffmpeg_path):     # 경로 존재하면 True, 아니면 False
    # 1.1 코덱을 PATH에 등록, 등록 여부 확인 후 등록
    if ffmpeg_path not in os.environ.get("PATH",""):    # 없으면 True => 등록
        os.environ["PATH"] = ffmpeg_path + os.pathsep + os.environ.get("PATH","")

    # 1.2 dll(실행파일) -> 운영체제에 등록(설치), 없으면 설치
    if hasattr(os,"add_dll_directory"):        # hasattr() 해당 변수를 가지고 있니? 내장함수
        try:
            os.add_dll_directory(ffmpeg_path)
            print("Codec Loading OK")
        except Exception as e:
            print(e)

Codec Loading OK


In [4]:
import torch


# CUDA check
def check_cuda():
    print(f"Transformers version: {transformers.__version__}")
    try:
        from transformers import AutoProcessor
        print("AutoProcessor imported successfully.")
    except ImportError as e:
        print(f"Error importing AutoProcessor: {e}")


    try:
        from transformers.utils import add_model_info_to_auto_map
        print("Internal utils (add_model_info_to_auto_map) imported successfully.")
    except ImportError as e:
        print(f"Error importing internal utils: {e}")
   
    print(f"Python version: {sys.version}")
    print(f"PyTorch version: {torch.__version__}")
   
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")
   
    if cuda_available:
        print(f"CUDA version: {torch.version.cuda}")
        device_count = torch.cuda.device_count()
        print(f"Device count: {device_count}")
       
        for i in range(device_count):
            print(f"Device {i}: {torch.cuda.get_device_name(i)}")
            print(f"  Capability: {torch.cuda.get_device_capability(i)}")
           
        current_device = torch.cuda.current_device()
        print(f"Current device index: {current_device}")
    else:
        print("WARNING: CUDA is not available. PyTorch will run on CPU.")

In [5]:
check_cuda()

Transformers version: 5.17.0
AutoProcessor imported successfully.
Error importing internal utils: cannot import name 'add_model_info_to_auto_map' from 'transformers.utils' (c:\big21\agentic_ai\venv\lib\site-packages\transformers\utils\__init__.py)
Python version: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
Device count: 1
Device 0: NVIDIA GeForce GTX 1660 SUPER
  Capability: (7, 5)
Current device index: 0


In [6]:
import os


import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [8]:
# GPU일 때는 16bit, CPU일 때는 32bit. gpu는 16bit일 때가 제일 빠름. cpu는 16bit일 때 작동 안 될 수 있음.
# Model => weight (기억)
# whisper-large-v3-turbo 기준: float16일 때 모델 크기가 1.6GB, float32일 때 모델 크기는 3.2GB
# GPU는 float16일 때 가장 빠름, CPU는 16bit 계산을 제대로 지원하지 않거나 느려질 수 있음. CPU에서는 float32 사용 필요
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [9]:
# 모델 다운로드하고 모델 오브젝트 생성
model_id = "openai/whisper-large-v3-turbo"  # decoder 32 layer -> 4 layer (turbo 빠르게 하려고). 번역 기능 사라짐. 변환 기능만 존재
model = AutoModelForSpeechSeq2Seq.from_pretrained(  # 모델이 없으면 다운로드 진행
    model_id,           # 다운로드
    dtype=torch_dtype,  # 모델의 가중치 크기 지정: 16bit(GPU)

    # low_cpu_mem_usage=True: 메모리 사용 최적화 => 빈 모델 먼저 만들고, 가중치를 채우는 대신 가중치를 읽으면서 바로 모델을 채워 CPU 메모리를 두 배로 쓰지 않게 하는 옵션 
    low_cpu_mem_usage=True,     
    use_safetensors=True    # 빠르게 모델 가중치 로딩 옵션
)

Loading weights: 100%|██████████| 587/587 [00:00<00:00, 2140.49it/s]


In [10]:
# (venv) C:\big21\agentic_ai>hf download openai/whisper-large-v3-turbo

In [11]:
# model의 weight를 gpu로 보내겠다.
model.to(device)

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1280)
      (layers): ModuleList(
        (0-31): 32 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (fc2): Linear(in_features=5120, out_features=1280, bias=Tr

In [12]:
# AutoProcessor: 모델에 맞는 특징 추출기(음성 -> 스펙트로그램), 토크나이저(토큰번호 <-> 글자)
processor=AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",     # 자동 음성 인식(ffmpeg)
    model=model,    # model 지정
    tokenizer=processor.tokenizer,      # 단어로 나옴.
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,  # 16bit : GPU에서 하니까
    device=device,      # GPU
    return_timestamps=True, # 문장 덩어리(chunk)별 시작, 끝 시간을 돌려준다. sec03 병합의 핵심
    chunk_length_s=10,    # 입력 오디오 10초씩 나누기.
    stride_length_s=2     # chunk 나눴을 때 경계 시간에 연결되는 단어가 있다면 앞 뒤가 겹치게 만들어 줘야 함.(앞 청크 2초, 뒤 청크 2초. 앞 뒤 총 4초가 겹침. 청크1: 0~10초, 청크2: 6~16) 
)

[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [13]:
sample=r"..\audio\lsy_audio_2023_58s.mp3"
# result = pipe(sample) 원본
result = pipe(sample, generate_kwargs={"language":"korean", "task":"transcribe"})
print(result)

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다. GPT-API에 대해서 생소하신 분들도 있을텐데 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챕터 프로그램을 만드는게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러가지가 될 수 있는데 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고', 'chunks': [{'timestamp': (0.0, 6.3), 'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.'}, {'timestamp': (7.18, 10.0), 'text': ' GPT-API에 대해서 생소하신 분들도 있을텐데'}, {'timestamp': (11.0, 17.0), 'text': ' 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서'}, {'timestamp': (17.0, 20.0), 'text': ' 우리가 원하는 프로그램을 어떻게 만드는지에 대해서'}, {'timestamp': (20.0, 22.0), 'text': ' 이야기할 거예요.'}, {'timestamp': (22.0, 24.0), 'text': ' 그래서 이런 강의들이 사실 많이 있습니다.'}, {'timestamp': (24.0, 27.48), 'text': ' 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면'}, {'timestamp': (27.48, 29.58), 'text': ' G

In [14]:
result.keys()

dict_keys(['text', 'chunks'])

In [15]:
result["text"]

' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다. GPT-API에 대해서 생소하신 분들도 있을텐데 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챕터 프로그램을 만드는게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러가지가 될 수 있는데 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고'

In [16]:
result["chunks"]
# stride 이후의 최종 결과물
# 초 간 빈 간격은 얘기를 멈춘 상황.

[{'timestamp': (0.0, 6.3),
  'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.'},
 {'timestamp': (7.18, 10.0), 'text': ' GPT-API에 대해서 생소하신 분들도 있을텐데'},
 {'timestamp': (11.0, 17.0), 'text': ' 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서'},
 {'timestamp': (17.0, 20.0), 'text': ' 우리가 원하는 프로그램을 어떻게 만드는지에 대해서'},
 {'timestamp': (20.0, 22.0), 'text': ' 이야기할 거예요.'},
 {'timestamp': (22.0, 24.0), 'text': ' 그래서 이런 강의들이 사실 많이 있습니다.'},
 {'timestamp': (24.0, 27.48), 'text': ' 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면'},
 {'timestamp': (27.48, 29.58), 'text': ' GPT로 명확한 미션을 달성하는'},
 {'timestamp': (29.58, 31.66), 'text': ' 챕터 프로그램을 만드는게 사실'},
 {'timestamp': (31.66, 34.32), 'text': ' 쉽지는 않은데 이걸 어떻게 해서'},
 {'timestamp': (34.32, 36.4), 'text': ' 구현을 하는지 그리고 그게 왜 필요한지에 대해서'},
 {'timestamp': (36.4, 37.34), 'text': ' 좀 이야기를 할 거고요.'},
 {'timestamp': (38.0, 40.0), 'text': ' 그 예제로'},
 {'timestamp': (40.0, 42.0), 'text': ' 예제는 여러가지가 될 수 있는데'},
 {'timestamp': (42.0, 45.66),
  'text': ' 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서'}

In [17]:
# Chunk -> dataframe -> CSV 파일 생성
import pandas as pd

start_end_text=[]   # 행들을 모을 빈 리스트

for chunk in result["chunks"]:  # chunk는 하나씩 꺼낸 dict.
    start, end = chunk["timestamp"]
    # start = chunk["timestamp"][0]   # (0.0, 6.3) 의 0.0
    # end = chunk["timestamp"][1]     # (0.0, 6.3) 의 6.3
    text = chunk["text"]
    start_end_text.append([start,end,text])
start_end_text

[[0.0, 6.3, ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.'],
 [7.18, 10.0, ' GPT-API에 대해서 생소하신 분들도 있을텐데'],
 [11.0, 17.0, ' 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서'],
 [17.0, 20.0, ' 우리가 원하는 프로그램을 어떻게 만드는지에 대해서'],
 [20.0, 22.0, ' 이야기할 거예요.'],
 [22.0, 24.0, ' 그래서 이런 강의들이 사실 많이 있습니다.'],
 [24.0, 27.48, ' 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면'],
 [27.48, 29.58, ' GPT로 명확한 미션을 달성하는'],
 [29.58, 31.66, ' 챕터 프로그램을 만드는게 사실'],
 [31.66, 34.32, ' 쉽지는 않은데 이걸 어떻게 해서'],
 [34.32, 36.4, ' 구현을 하는지 그리고 그게 왜 필요한지에 대해서'],
 [36.4, 37.34, ' 좀 이야기를 할 거고요.'],
 [38.0, 40.0, ' 그 예제로'],
 [40.0, 42.0, ' 예제는 여러가지가 될 수 있는데'],
 [42.0, 45.66, ' 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서'],
 [45.66, 47.1, ' 생성하는 프로그램을 만드는 것을'],
 [47.1, 48.46, ' 다루려고 합니다.'],
 [49.84, 51.96, ' 프로그램이 실행되는 모습을 한번 보여드릴게요.'],
 [52.84, 58.0, ' 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고']]

In [19]:
# Dataframe
df = pd.DataFrame(
    data=start_end_text,
    columns=["start","end","text"]
)
df

,start,end,text
0,0.00,6.30,안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.
1,7.18,10.00,GPT-API에 대해서 생소하신 분들도 있을텐데
2,11.00,17.00,"우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서"
3,17.00,20.00,우리가 원하는 프로그램을 어떻게 만드는지에 대해서
4,20.00,22.00,이야기할 거예요.
5,22.00,24.00,그래서 이런 강의들이 사실 많이 있습니다.
6,24.00,27.48,그래서 여러가지들이 있는데 이 강의 특징이라고 한다면
7,27.48,29.58,GPT로 명확한 미션을 달성하는
8,29.58,31.66,챕터 프로그램을 만드는게 사실
9,31.66,34.32,쉽지는 않은데 이걸 어떻게 해서


In [20]:
# 파일 저장
df.to_csv(
    "lsy_audio_2023_58.csv",
    index=False, # index 저장x. index 의미 없는 경우
    sep="|"     # 구분자, default=','
)